# 04 · Extensions: Table 3, SAE construction, three-class encoders, five seeds

Later runs, on top of `results/main/` (notebook 02) and `results/fedlex/` (notebook 03).
Results: `results/extensions/` (the seeds 3–4 runs in `extensions_seeds_3_4/`, merged into `raw/five_seeds/`).

| Section | Feeds |
|---|---|
| 1. SAE-free random directions | Table 3's reference check (row A3's published draw) |
| 2. Band-matched random SAE latents | Figure 1 (random curve), Table 3 row A2's reference |
| 3. Table 3 on six settings | Table 3, the erasure-extent table |
| 4. SAE construction | The SAE-construction table |
| 5. Ten-encoder study, three-class settings | The ten-encoder table |
| 6. Displacement-matched PCA in frozen transfer | The frozen-transfer figure |
| 7. Seeds 3–4 and the five-seed merge | Table 2, the extended in-distribution and erasure-extent tables; Figures 1 and 2, the frozen-transfer figure |
| 8. The rank sweep on seeds 3–4 | The frozen-transfer figure, panel C |

Every step is cached. Roughly 40 hours on a Colab GPU in total; sections 3, 5 and 7 take most of it.

In [ ]:
# On Colab: mount Google Drive and move to your copy of the repository (see the README: "Running on Google Colab";
# set REPO if your copy is elsewhere). Skipped outside Colab.
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/label-free-deconfounding-embeddings'
    %cd {REPO}
    %pip install -q -r requirements.txt
except ImportError:
    print('not on Colab: using the current working directory')

In [ ]:
import os, sys
if not os.path.isdir('code') and os.path.isdir(os.path.join('..', 'code')):
    os.chdir('..')                                   # opened from notebooks/
assert os.path.isdir('code') and os.path.isdir('data'), f'run from the repository root; cwd={os.getcwd()}'
sys.path.insert(0, 'code')

import torch                                         # first: some modules load diptest, whose macOS wheel bundles libomp
import paths

# QUICK = True runs every section end to end in minutes on a subset (two seeds, a 20-epoch SAE, a few settings),
# writing to results/*_quick/ so it never mixes with the real results. Its numbers are NOT the paper's.
QUICK = False
paths.configure(quick_run=QUICK)                     # before the experiment modules are imported
DEVICE = 'cuda' if torch.cuda.is_available() else None   # None: each step's own choice
print('cwd:', os.getcwd(), '| device:', DEVICE or 'auto', '| QUICK' if QUICK else '')

In [ ]:
import artifacts, experiments as E, ood
import revised as R
import alternatives as A
import focused_additions as FA
import focused_extensions as FX

artifacts.set_results_dir(paths.EXTENSIONS)
E.set_embed_cache(paths.EMBED_CACHE)
ood.PROMPT_STYLE = 'parent'
print('earlier runs read from', FA.PUBLISHED_ROOTS, '| seeds 3-4 root:', FX.extension_root())

TABLE3_SETTINGS, FIG1_SETTINGS = FA.TABLE3_SETTINGS, FA.FIG1_SETTINGS    # Table 3's two; Figure 1's four
DRAWS = FA.BAND_DRAWS                 # 5 random subsets per SAE
NONLINEAR = ('mlp',)                  # Table 3 reports linear AUC, MLP AUC and R@10
T3X_CELLS, T3X_SEEDS = FX.T3X_CELLS, FX.T3X_SEEDS          # six (setting, encoder) cells, seeds 0-4
SAE_SETTINGS, SAE_CONFIGS, SAE_SEEDS = FX.SAE_SETTINGS, FX.SAE_CONFIGS, FX.SAE_SEEDS
ENC3_SETTINGS, ENC3_ENCODERS = FX.ENC3_SETTINGS, R.ENCODER_STUDY        # all ten encoders
TRANSFER_PAIRS, SEED_JOBS, SEED_KW, N_BOOT = FX.TRANSFER_3CLASS, FX.SEED_JOBS, {}, 10000
SEEDS, OVERRIDES = R.SWEEP_SEEDS, {}
if QUICK:            # needs QUICK runs of notebooks 02 and 03 (results/*_quick)
    TABLE3_SETTINGS = FIG1_SETTINGS = ('swiltra_de_fr_it',)
    DRAWS, NONLINEAR, SEEDS, OVERRIDES, N_BOOT = 2, (), (0, 1), dict(n_epochs=20), 200
    T3X_CELLS, T3X_SEEDS = (('swiltra_de_fr_it', 'e5-small', 'full'),), (0,)
    SAE_SETTINGS, SAE_CONFIGS, SAE_SEEDS = (('swiltra_de_fr_it', 'e5-small'),), FX.SAE_CONFIGS[:1], (0,)
    ENC3_SETTINGS, ENC3_ENCODERS = ('swiltra_de_fr_it',), ('mini',)
    TRANSFER_PAIRS = (('e5-small', 'swiltra_de_fr_it'),)
    SEED_KW = dict(seeds=(3,), fig1_settings=FIG1_SETTINGS, fig2_rows=(('swiltra_de_fr_it', 'e5-small'),),
                   headline=(('cap', 'e5-small'),), donors=(('e5-small', 'cap'),))

## 1. SAE-free random directions

Directions-LEACE on 32 unit-norm Gaussian directions, one draw per seed, on Table 3's two settings, written next to
the main results (`results/main/raw/random_directions/`), where the Table 3 runs' reference check reads them.

In [ ]:
for key in TABLE3_SETTINGS:
    FA.run_r1(key, 'e5-small', seeds=SEEDS, budgets=(32,), draws=1, nonlinear=NONLINEAR, sae_arms=False,
              references=False, device=DEVICE, fit_device=DEVICE, **OVERRIDES)
    try:
        FA.export_table3_slot(key, 'e5-small')
    except FileExistsError as err:
        print('already written:', err)

## 2. Band-matched random SAE latents

Directions-LEACE on N decoder rows drawn uniformly from the latents MI scores (training firing rate in [0.02, 0.98]),
five draws per seed, on the coverage runs' splits and SAEs (each MI selection is reproduced first).

In [ ]:
for key in FIG1_SETTINGS:
    FA.run_band_control(key, 'e5-small', seeds=SEEDS, budgets=None if not QUICK else (8, 32), draws=DRAWS,
                        nonlinear=NONLINEAR, fit_device=DEVICE, **OVERRIDES)

## 3. Table 3 on six settings

One SAE per split, refitted here; on it: MI-selected latents (A1), random band latents (A2), Gaussian directions (A3)
and displacement-matched Gaussian directions (A4); Activation-LEACE (B1), orthogonal removal with and without the
mean (B2, B3) and orthogonal removal of span(ΣD_Sᵀ) (B4); displacement-matched PCA (C1). Every seed is checked
against the published fit of the same seed: the rows without an SAE must reproduce it.

In [ ]:
for key, enc, arms in T3X_CELLS:
    FX.run_table3x(key, enc, seeds=T3X_SEEDS, arms=arms, draws=DRAWS, nonlinear=NONLINEAR, fit_device=DEVICE,
                   **OVERRIDES)

## 4. SAE construction

Table 3's core rows with other TopK dictionaries (expansion factor and k). A configuration whose firing band holds
fewer than N = 32 latents is skipped and recorded (`raw/table3x/sae_configs_skipped.json`).

In [ ]:
FX.run_sae_configs(settings=SAE_SETTINGS, configs=SAE_CONFIGS, seeds=SAE_SEEDS, draws=DRAWS, nonlinear=NONLINEAR,
                   fit_device=DEVICE, **OVERRIDES)

## 5. Ten-encoder study on the three-class settings (the ten-encoder table)

Each cell exactly as notebook 02 §5 runs it, on SwiLTra DE–FR–IT and SCOTUS Oyez–Lexis–Wikipedia. Seven encoders
were run first and the other three later (on a CPU); the cells are identical in recipe, splits and audit.

In [ ]:
FX.run_encoders_3class(settings=ENC3_SETTINGS, encoders=ENC3_ENCODERS, seeds=SEEDS if QUICK else R.ENCODER_SEEDS,
                       device=DEVICE, **OVERRIDES)

## 6. Displacement-matched PCA in frozen transfer

For the three-class donors: PCA removing the fewest leading components whose displacement of the donor's training
embeddings reaches that seed's SAE-LEACE map, frozen and transferred like the other maps (seeds 0–2).

In [ ]:
FX.run_transfer_matched(pairs=TRANSFER_PAIRS, seeds=(0,) if QUICK else R.OOD_SEEDS, device=DEVICE)
TM = FX.transfer_matched_summary(pairs=SEED_KW.get('donors', FX.TRANSFER_DONORS), n_boot=N_BOOT)

## 7. Seeds 3–4 and the five-seed merge

The cells that were run on seeds 0–2 get seeds 3–4, each with the published run's settings, under
`extensions_seeds_3_4/`: Figure 1's coverage and band runs, Figure 2's geometry runs, Table 2's three-seed cells
(SemEval, CAP) and the transfer donors of the frozen-transfer figure. `merge_five_seeds` then writes the merged files to
`raw/five_seeds/` after checking that the two batches share the recipe, split policy and audit.

In [ ]:
FX.run_seed_extension(jobs=SEED_JOBS, device=DEVICE, fit_device=DEVICE, quick=QUICK, **SEED_KW, **OVERRIDES)
FX.merge_five_seeds(jobs=SEED_JOBS, strict=not QUICK, **{k: v for k, v in SEED_KW.items() if k != 'seeds'})
TM5 = FX.transfer_matched_summary(pairs=SEED_KW.get('donors', FX.TRANSFER_DONORS), five=True, n_boot=N_BOOT)

## 8. The rank sweep on seeds 3–4 (the frozen-transfer figure, panel C)

PCA at ranks 1–32 for the original transfer donors on seeds 3–4, in a results folder of its own (under the main one,
the runner would extend the published files in place). `paper-artifacts/build.py` merges it with seeds 0–2.

In [ ]:
artifacts.set_results_dir(os.path.join(paths.EXTENSIONS, 'rank_sweep_seeds_3_4'))
try:
    matrix = [('e5-small', 'cap')] if QUICK else R.OOD_MATRIX
    FX.run_rank_sweep_extension(seeds=(3,) if QUICK else FX.EXT_SEEDS, matrix=matrix, device=DEVICE)
finally:
    artifacts.set_results_dir(paths.EXTENSIONS)